# AutoEncoder Anomaly Detection

An autoencoder is a neural network trained to reproduce its input through a narrow bottleneck. It learns the patterns most observations share, so points that break those patterns are reconstructed badly. The **reconstruction error** is the anomaly score.

This tutorial needs OutlierDetectionNetworks.jl, OutlierDetection.jl and Flux.jl.

In [ ]:
Pollis.packages("OutlierDetection", "OutlierDetectionNetworks", "Flux", "Plots", "StatsBase")

In [ ]:
using OutlierDetection, OutlierDetectionNetworks, Flux
using OutlierDetectionNetworks.Templates: MLPAutoEncoder
using Plots, Statistics, StatsBase, Random

Random.seed!(1);

## Data with Hidden Structure

Four features driven by one hidden variable t. The anomalies have ordinary values in every single feature, but they break the relationship between features, so no per-feature rule can find them.

In [ ]:
t = randn(300)
normal = hcat(t, t .^ 2, sin.(2 .* t), -t) .+ 0.1 .* randn(300, 4)
tt = randn(10)
anomalies = hcat(tt, -tt .^ 2 .+ 1, cos.(2 .* tt), tt) .+ 0.1 .* randn(10, 4)
X = vcat(normal, anomalies)
is_true = vcat(falses(300), trues(10))

scatter(X[:, 1], X[:, 2]; group = ifelse.(is_true, "Anomaly", "Normal"),
    markerstrokewidth = 0, xlabel = "Feature 1", ylabel = "Feature 2")

## Scale the Data

The decoder ends in a sigmoid, which outputs values between 0 and 1. Scale every feature to [0, 1] and put observations in columns.

In [ ]:
lo, hi = minimum(X, dims = 1), maximum(X, dims = 1)
Xs = Float32.(permutedims((X .- lo) ./ (hi .- lo)))
size(Xs)

## Fit the AutoEncoder

`MLPAutoEncoder(inputs, latent, hidden)` builds an encoder with hidden layers of 32 and 16 units down to a 2-dimensional bottleneck, and a mirrored decoder.

In [ ]:
encoder, decoder = MLPAutoEncoder(size(Xs, 1), 2, [32, 16]; bias = true)
detector = AEDetector(encoder = encoder, decoder = decoder, epochs = 50, opt = Adam(1e-3))
model, scores = OutlierDetection.fit(detector, Xs; verbosity = 0)

threshold = quantile(scores, 1 - 0.05)
flagged = scores .> threshold
println("Anomalies found: ", sum(flagged .& is_true), " of ", sum(is_true))

Expect some anomalies to be missed: with a few hundred observations, a neural detector is noisy. More epochs, a different latent size or a proximity baseline are worth trying.

In [ ]:
histogram(scores; bins = 40, xlabel = "Reconstruction error", ylabel = "Count", label = "")
vline!([threshold]; color = :red, linestyle = :dash, label = "Top 5%")

## Which Features Are Unusual?

The reconstruction error splits by feature. Put the network in test mode (so its normalisation layers use their stored statistics) and compare each input with its reconstruction.

In [ ]:
Flux.testmode!(model.chain)
top = sortperm(scores; rev = true)[1:5]
err = (model.chain(Xs)[:, top] .- Xs[:, top]) .^ 2
round.(permutedims(err), digits = 3)   # rows: top 5 points, columns: features

## Score New Data

New observations must be scaled with the **training** minimum and maximum.

In [ ]:
X_new = vcat(hcat(0.5, 0.25, sin(1.0), -0.5), hcat(0.5, -0.25, cos(1.0), 0.5))
Xn = Float32.(permutedims((X_new .- lo) ./ (hi .- lo)))
new_scores = OutlierDetection.transform(detector, model, Xn)
println("New scores: ", round.(new_scores, digits = 4), "   threshold: ", round(threshold, digits = 4))

## Seed Stability

The random initial weights change the result. Refit with another seed and compare. The ordering of ordinary points is mostly noise, so the full-ranking correlation can be modest; what matters is whether the **top** points agree.

In [ ]:
Random.seed!(2)
enc2, dec2 = MLPAutoEncoder(size(Xs, 1), 2, [32, 16]; bias = true)
_, scores2 = OutlierDetection.fit(AEDetector(encoder = enc2, decoder = dec2, epochs = 50, opt = Adam(1e-3)), Xs; verbosity = 0)
println("Spearman correlation between seeds: ", round(corspearman(scores, scores2), digits = 3))
top16(s) = sortperm(s; rev = true)[1:16]
println("Shared points in the two top-16 lists: ", length(intersect(top16(scores), top16(scores2))))

## Summary
- The autoencoder learns the shared structure; the reconstruction error is the anomaly score.
- Scale features to [0, 1] and apply the training scaling to new data.
- The per-feature error shows which features a point gets wrong.
- Check stability across random seeds before trusting the ranking.